# Policy gradients with PyTorch
**Machine Learning · Week 7 (Reinforcement learning) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/ml/w7d5.html

**Policy gradients:** instead of learning values and acting greedily, learn the policy itself, a neural
network that outputs action probabilities, and push it towards actions that led to high returns.
The task: the classic **cart-pole**. Push a cart left or right to keep a pole upright.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_num_threads(1)

class CartPole:
    """The classic cart-pole (Barto, Sutton and Anderson, 1983). State: cart position and velocity,
    pole angle and angular velocity. Reward +1 per step; the episode ends when the pole tips past
    12 degrees, the cart leaves the track, or after 200 steps (4 seconds)."""
    g, m_cart, m_pole, half_length, force, dt = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02

    def __init__(self, rng, max_steps=200):
        self.rng, self.max_steps = rng, max_steps

    def reset(self):
        self.state, self.t = self.rng.uniform(-0.05, 0.05, 4), 0
        return self.state.copy()

    def step(self, action):                       # action 0: push left, 1: push right
        x, x_dot, theta, theta_dot = self.state
        f = self.force if action == 1 else -self.force
        cos, sin = np.cos(theta), np.sin(theta)
        total = self.m_cart + self.m_pole
        temp = (f + self.m_pole * self.half_length * theta_dot ** 2 * sin) / total
        theta_acc = (self.g * sin - cos * temp) / (self.half_length * (4 / 3 - self.m_pole * cos ** 2 / total))
        x_acc = temp - self.m_pole * self.half_length * theta_acc * cos / total
        self.state = np.array([x + self.dt * x_dot, x_dot + self.dt * x_acc,
                               theta + self.dt * theta_dot, theta_dot + self.dt * theta_acc])
        self.t += 1
        fell = abs(self.state[0]) > 2.4 or abs(self.state[2]) > 12 * np.pi / 180
        return self.state.copy(), 1.0, fell or self.t >= self.max_steps

env = CartPole(np.random.default_rng(0))
lengths = []
for episode in range(20):
    state, done, n = env.reset(), False, 0
    while not done:
        state, reward, done = env.step(int(np.random.default_rng(episode * 1000 + n).integers(2)))
        n += 1
    lengths.append(n)
print("random pushing keeps the pole up for", np.mean(lengths), "steps on average (200 is the maximum)")

## The policy network
Four numbers in (the state), two scores out, softmax gives the probabilities of pushing left or right.

In [ ]:
torch.manual_seed(3)
policy = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.Tanh(), torch.nn.Linear(32, 2))
state = torch.tensor(env.reset(), dtype=torch.float32)
probs = torch.softmax(policy(state), dim=0)
print("P(push left), P(push right) for a starting state:", probs.detach().numpy().round(3))
action = torch.distributions.Categorical(probs=probs).sample()
print("sampled action:", ["left", "right"][int(action)])

## REINFORCE
Play an episode, compute the return $G_t$ after every step, then make each action more likely in
proportion to how good its return was (compared with the episode's average):
loss $= -\sum_t \log \pi_\theta(a_t \mid s_t)\,\hat G_t$.

In [ ]:
def train(episodes=400, lr=0.005, gamma=0.99, seed=3):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    env = CartPole(rng)
    policy = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.Tanh(), torch.nn.Linear(32, 2))
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    lengths, checkpoints = [], {0: [p.detach().clone() for p in policy.parameters()]}
    for episode in range(1, episodes + 1):
        state, done, log_probs, rewards = env.reset(), False, [], []
        while not done:
            dist = torch.distributions.Categorical(logits=policy(torch.tensor(state, dtype=torch.float32)))
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            state, reward, done = env.step(int(action))
            rewards.append(reward)
        G, returns = 0.0, []
        for r in reversed(rewards):                       # returns, from the end backwards
            G = r + gamma * G
            returns.append(G)
        returns = torch.tensor(returns[::-1])
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)   # better than average: +, worse: -
        loss = -(torch.stack(log_probs) * returns).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lengths.append(len(rewards))
        if episode in (50, 150, episodes):
            checkpoints[episode] = [p.detach().clone() for p in policy.parameters()]
    return policy, lengths, checkpoints

policy, lengths, checkpoints = train()
smooth = np.convolve(lengths, np.ones(20) / 20, mode="valid")
plt.plot(lengths, color="#ffc7ae", lw=1, label="each episode")
plt.plot(np.arange(19, len(lengths)), smooth, color="#ff8a50", lw=2.5, label="average of 20")
plt.xlabel("episode")
plt.ylabel("steps the pole stayed up")
plt.title("REINFORCE learns to balance")
plt.legend()
plt.show()

## Test the trained policy
Act greedily (always the more likely push) and allow up to 1,000 steps: 20 seconds of balancing.

In [ ]:
test_env = CartPole(np.random.default_rng(42), max_steps=1000)
results = []
for _ in range(10):
    state, done, n = test_env.reset(), False, 0
    while not done:
        with torch.no_grad():
            action = int(policy(torch.tensor(state, dtype=torch.float32)).argmax())
        state, _, done = test_env.step(action)
        n += 1
    results.append(n)
print("steps balanced in 10 test runs (max 1000):", results)

## Your turn: what did the network learn?
Ask the trained policy for $P(\text{push right})$ in a few hand-made states (cart at rest in the middle):
the pole leaning 5° left, upright, and 5° right, all with zero angular velocity. Then a tricky one: leaning
3° right, but already swinging back to the left at 1 radian per second. State order:
(cart position, cart velocity, pole angle in radians, pole angular velocity).

In [ ]:
def p_right(x, x_dot, theta, theta_dot):
    return 0.5          # TODO: softmax of policy(torch.tensor([...], dtype=torch.float32)), take entry 1

deg = np.pi / 180
for name, s in [("leaning left", (0, 0, -5 * deg, 0)), ("upright", (0, 0, 0, 0)), ("leaning right", (0, 0, 5 * deg, 0)),
                ("right, swinging back left", (0, 0, 3 * deg, -1.0))]:
    print(f"{name:26s} P(push right) = {p_right(*s):.3f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def p_right(x, x_dot, theta, theta_dot):
    with torch.no_grad():
        return float(torch.softmax(policy(torch.tensor([x, x_dot, theta, theta_dot], dtype=torch.float32)), dim=0)[1])

deg = np.pi / 180
for name, s in [("leaning left", (0, 0, -5 * deg, 0)), ("upright", (0, 0, 0, 0)), ("leaning right", (0, 0, 5 * deg, 0)),
                ("right, swinging back left", (0, 0, 3 * deg, -1.0))]:
    print(f"{name:26s} P(push right) = {p_right(*s):.3f}")
print("It pushes towards the side the pole is falling to, and it watches the swing (the angular velocity)")
print("even more than the angle: move the cart under where the pole is going, not where it is.")